> 在线讲义：[第 24 章 · 根据架构选择加速器：先估下界，再找差距，最后算账](https://wikieden.github.io/ai-infra-from-constraints/lessons/24-choose-accelerator.html) · 本书《从约束推导 AI Infra》

# 第 24 章 · 根据架构选择加速器：先估下界，再找差距，最后算账

配套讲义：仓库里 `ai-infra/notes/lessons/24-choose-accelerator.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 阶段 | $\sum \max$ | `staged_time(stages)` | 20 μs vs 合计 11 μs |
| 第一步 转折 | $I^* d / (d - 2I^*)$ | `turning_rows(165.2e12, 1.008e12)` | 178.1 |
| 第二步 设备 | $V/R$、$F/P$ | `device_table()` | 16.22 / 9.12 ms；7.1 / 57.0 / 39.1 / 68.5% |
| 第四步 成本 | $c_A/c_B < t_B/t_A$ | `cheaper(...)` | |
| 第四步 回本 | $F_0/\Delta c$ | `breakeven(20e6, 0.5, 100, 10_000, 0.5)` | $4 \times 10^{13}$、254 台、508 台 |

资源时间模型、Roofline 转折、Qwen3-8B 设备比较、vLLM 实测、能耗与例 4-3 取自 AII 第 4 章 §4.8。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
DEVICES = {
    "RTX 3090": (24, 936, 71.0),
    "RTX 4090": (24, 1008, 165.2),
    "RTX 5090": (32, 1792, 209.5),
    "RTX PRO 6000": (96, 1792, 503.8),
    "A100": (80, 2039, 312.0),
    "H100": (80, 3350, 989.4),
    "H200": (141, 4800, 989.5),
}
DECODE_BYTES = 16.345e9  # Qwen3-8B BF16 单请求 8K 每步权重 + KV
PREFILL_FLOPS = 61.85e12  # 4K prefill

**函数 `staged_time`**：阶段内各资源并行（取最大），阶段之间串行。返回 (总时间, 各资源合计)。

In [ ]:
def staged_time(stages: list[dict[str, float]]) -> tuple[float, dict[str, float]]:
    """阶段内各资源并行（取最大），阶段之间串行。返回 (总时间, 各资源合计)。"""
    total = sum(max(s.values()) for s in stages)
    per_res: dict[str, float] = {}
    for s in stages:
        for r, t in s.items():
            per_res[r] = per_res.get(r, 0) + t
    return total, per_res

**函数 `projection_us`**：(矩阵时间 μs, 片外读写 μs)：BF16 输入 + 权重读一次、输出写一次。

In [ ]:
def projection_us(m: int, p: float, r: float, d: int = 4096) -> tuple[float, float]:
    """(矩阵时间 μs, 片外读写 μs)：BF16 输入 + 权重读一次、输出写一次。"""
    return 2 * m * d * d / p * 1e6, (2 * m * d + d * d) * 2 / r * 1e6

**函数 `turning_rows`**：I(M) = Md/(d+2M) 等于 P/R 时的 M*。

In [ ]:
def turning_rows(p: float, r: float, d: int = 4096) -> float:
    """I(M) = Md/(d+2M) 等于 P/R 时的 M*。"""
    i = p / r
    return i * d / (d - 2 * i)

**函数 `decode_flops`**：小工具。

In [ ]:
def decode_flops() -> float:
    w_layer = 2 * 4096**2 + 2 * 4096 * 1024 + 3 * 4096 * 12288
    return 2 * 36 * w_layer + 4 * 36 * 8192 * 4096 + 1.245e9

**函数 `device_table`**：每张卡 (decode 读取下界 ms, 4K prefill 矩阵时间 ms)。

In [ ]:
def device_table() -> dict[str, tuple[float, float]]:
    """每张卡 (decode 读取下界 ms, 4K prefill 矩阵时间 ms)。"""
    return {k: (DECODE_BYTES / (bw * 1e9) * 1e3, PREFILL_FLOPS / (p * 1e12) * 1e3) for k, (_, bw, p) in DEVICES.items()}

**函数 `cheaper`**：按运行时间计费，A 每任务更便宜 ⇔ c_A/c_B < t_B/t_A。

In [ ]:
def cheaper(c_a: float, t_a: float, c_b: float, t_b: float) -> bool:
    """按运行时间计费，A 每任务更便宜 ⇔ c_A/c_B < t_B/t_A。"""
    return c_a / c_b < t_b / t_a

**函数 `breakeven`**：(回本 token 数, 寿命内产出, 寿命内回本所需最少机器数)。

In [ ]:
def breakeven(f0: float, saving_per_m: float, machines: int, tok_s: float, util: float, years: float = 1) -> tuple[float, float, int]:
    """(回本 token 数, 寿命内产出, 寿命内回本所需最少机器数)。"""
    need = f0 / (saving_per_m / 1e6)
    per_machine = tok_s * util * 365 * 86400 * years
    return need, machines * per_machine, math.ceil(need / per_machine)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：资源相加、阶段取最大")
total, per = staged_time([{"matrix": 10, "memory": 1}, {"matrix": 1, "memory": 10}])
print(f"  两阶段 {total:.0f} μs，各资源合计 {per}（只看合计会估成 {max(per.values()):.0f} μs）")
print(f"  4090 算 8.59 GFLOPs：FP16 累加 {8.59e9 / 330e12 * 1e6:.0f} μs，FP32 累加 {8.59e9 / 165.2e12 * 1e6:.0f} μs")
for m in (1, 256):
    c, v = projection_us(m, 165.2e12, 1.008e12)
    print(f"  M = {m:3d}：矩阵 {c:.2f} μs，读写 {v:.1f} μs")
print(f"  I* = {165.2e12 / 1.008e12:.0f} FLOPs/B，M* = {turning_rows(165.2e12, 1.008e12):.1f}")

In [ ]:
print("\n第二步：同一个 Qwen3-8B 比较设备")
print(f"  decode 一步 {decode_flops() / 1e9:.2f} GFLOPs，读 {DECODE_BYTES / 1e9} GB")
tab = device_table()
for k, (dec, pre) in tab.items():
    print(f"  {k:13s} decode {dec:6.2f} ms（{1000 / dec:5.1f} token/s），prefill {pre:6.1f} ms")
for a, b in (("RTX 3090", "RTX 4090"), ("A100", "H100")):
    print(f"  {a}→{b}：读取 −{(1 - tab[b][0] / tab[a][0]) * 100:.1f}%，矩阵 −{(1 - tab[b][1] / tab[a][1]) * 100:.1f}%")
print(f"  235B-A22B 每步 13.697 GB：H200 {13.697e9 / 4.8e12 * 1e3:.2f} ms（{4.8e12 / 13.697e9:.1f} token/s）")

In [ ]:
print("\n第三步：预测和实测")
lb = DECODE_BYTES / 1.792e12 * 1e3
print(f"  PRO 6000 下界 {lb:.2f} ms，实测 25.83 ms → MBU {lb / 25.83 * 100:.1f}%，多出 {25.83 - lb:.1f} ms")
print(f"  batch 64、2K：读 34.46 GB 下界 {34.46e9 / 1.792e12 * 1e3:.2f} ms，实测 29.63 ms → {34.46e9 / 29.63e-3 / 1e12:.2f} TB/s")

In [ ]:
print("\n第四步：成本、能耗与回本")
for t_us in (42.9, 32.6):
    print(f"  600 W × {t_us} μs = {600 * t_us * 1e-3:.1f} mJ")
print(f"  Mac 持平功率：单行 {600 * 42.9 / 166:.0f} W，256 行 {600 * 32.6 / 1835:.1f} W")
print(f"  每步能耗：4090 {450 * tab['RTX 4090'][0] / 1e3:.2f} J，5090 {575 * tab['RTX 5090'][0] / 1e3:.2f} J")
need, out, n = breakeven(20e6, 0.5, 100, 10_000, 0.5)
print(f"  例 4-3：回本 {need:.1e} token，一年产出 {out:.2e}，至少 {n} 台；半年 {breakeven(20e6, 0.5, 100, 10_000, 0.5, 0.5)[2]} 台")

## 3. 代码里才看得到的

1. `decode_flops` 的每层参数 $2 \cdot 4096^2 + 2 \cdot 4096 \cdot 1024 + 3 \cdot 4096 \cdot 12288$：K、V 只有 8 个头 × 128 = 1,024 宽（第 9 章 GQA）。
2. 600 W × 42.9 μs 代码得 25.7 mJ，原文 25.8 mJ：原文用的是未取整的实测时间。
3. 回本的机器数用上取整：$4 \times 10^{13} / 1.577 \times 10^{11} = 253.7$，所以是 254 台。

## 4. 自己改一改

- 给 `DEVICES` 加一行 B200（180 GB、8 TB/s、约 2.25 PFLOP/s BF16 稠密），看 decode 和 prefill 各排第几。
- 把 `staged_time` 的第二阶段改成可以和第一阶段部分重叠，总时间怎么变？
- 改 `breakeven` 的利用率，画出机器数随利用率的曲线。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 同一个灶相加，灶和烤箱取最大，汤等肉就串着来；
> 堵车看起步（带宽），长途看马力（矩阵）；
> 导航不准先改路，再数红灯；
> 贵车快一倍就不贵，专车要跑够趟数才回本。